# 1.3 - Math Foundations: Transformer Architecture

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook builds the transformer's core math from the matrix up: a hand-run matmul, scaled dot-product attention written out by hand, the three activation functions every modern block uses, a demo of why activations are non-negotiable, and finally a full multi-head attention block wired end to end in PyTorch. Everything is a small runnable experiment - shapes you can trace, numbers you can check by hand.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install the numerics stack

The whole notebook runs on two libraries: NumPy for plain array math and PyTorch for tensors, autograd, and the built-in attention/activation ops. One quiet install line and we're done.

In [ ]:
!pip install -q numpy torch

**Explanation**

Environment prep, not logic - it pulls the two packages the rest of the notebook imports. On Colab both are usually preinstalled, so this is a fast no-op safety net.

**How the code works - step by step**
- **`!pip install -q numpy torch`** - installs NumPy and PyTorch quietly (`-q` suppresses the progress spam).

**In one line:** make sure `numpy` and `torch` are importable.

**What you'll see:** (no output - environment prep)

## Setup - imports

Pull in the exact handful of PyTorch pieces the notebook leans on, plus Python's `math` for the one scalar square root attention needs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

**Explanation**

Pure import cell - it names the tools everything downstream uses. `torch` is the tensor engine, `nn` gives layer classes, `F` gives the functional ops (softmax, relu, attention), and `math` supplies `sqrt` for the attention scaling constant.

**How the code works - step by step**
- **`import torch`** - the core tensor library.
- **`import torch.nn as nn`** - layer building blocks (`Linear`, `Sequential`, `GELU`, `Module`).
- **`import torch.nn.functional as F`** - stateless ops: `relu`, `gelu`, `silu`, `softmax`, `scaled_dot_product_attention`.
- **`import math`** - just for `math.sqrt` to scale attention scores.

**In one line:** load tensors, layers, functional ops, and a square root.

**What you'll see:** (no output - environment prep)

## 1 - Matmul at scale

Every layer in a transformer is a matrix multiply at heart, so start by running one by hand. A 3x4 times a 4x2 gives a 3x2 - the inner dimensions (4) cancel, the outer dimensions (3, 2) survive. Getting fluent at reading that shape rule is the whole game later.

In [ ]:
A = torch.tensor([[1.,2.,1.,0.], [0.,1.,2.,3.], [2.,0.,1.,1.]])
B = torch.tensor([[1.,2.], [0.,1.], [3.,0.], [1.,1.]])
C = A @ B
print(C)

**Explanation**

A shape-and-arithmetic warm-up, not a model - it multiplies two concrete matrices with the `@` operator so you can verify the result cell by cell. The point is the dimension bookkeeping: `(3,4) @ (4,2) -> (3,2)`.

**How the code works - step by step**
- **`A`** - a 3x4 tensor (3 rows, 4 columns).
- **`B`** - a 4x2 tensor; its 4 rows match A's 4 columns, so the multiply is legal.
- **`C = A @ B`** - the matrix product; each entry is a dot product of an A-row with a B-column.
- **`print(C)`** - shows the resulting 3x2 matrix.

**In one line:** inner dims must match, outer dims give the output shape.

**What you'll see:** A 3x2 tensor printed - the first row is `[4., 4.]` (from A's row `[1,2,1,0]` dotted with each B column), and two more rows below it.

## 2 - Scaled dot-product attention, by hand

This is the single most important formula in the course: `softmax(QK^T / sqrt(d_k)) V`. Queries ask, keys answer, the dot product scores the match, dividing by `sqrt(d_k)` keeps the scores from blowing up, softmax turns them into weights, and those weights blend the values. Writing it out yourself - instead of calling a library - is how it stops being a black box.

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

torch.manual_seed(0)
Q = torch.randn(3, 4)
K = torch.randn(3, 4)
V = torch.randn(3, 4)
out = attention(Q, K, V)
print(f'output shape: {out.shape}')
weights = torch.softmax(Q @ K.T / math.sqrt(4), dim=-1)
print(f'attn weights row 0: {weights[0].round(decimals=3)}')

**Explanation**

A from-scratch implementation of the attention equation plus a quick sanity run. `attention` is a pure function: score every query against every key, optionally mask, normalize with softmax, then take the weighted sum of values. The extra lines re-derive the weights so you can confirm each attention row sums to 1.

**How the code works - step by step**
- **`d_k = Q.shape[-1]`** - the key/query dimension used for scaling.
- **`scores = Q @ K.transpose(-2,-1) / sqrt(d_k)`** - all pairwise query-key dot products, scaled down so softmax stays in a sane range.
- **`masked_fill(mask == 0, -inf)`** - optional: sets forbidden positions to -inf so softmax zeroes them (used for causal masking).
- **`weights = softmax(scores, dim=-1)`** - turns each row of scores into a probability distribution over keys.
- **`weights @ V`** - the output: each token's value vector is a weighted blend of all value vectors.
- **`torch.manual_seed(0)` + random Q, K, V** - fixed-seed 3x4 inputs so the run is reproducible.

**In one line:** score, scale, softmax, then weighted-sum the values.

**What you'll see:** Two prints: `output shape: torch.Size([3, 4])` (same shape as V) and `attn weights row 0: ...` - three numbers that are all non-negative and sum to 1.0.

## 3 - Activation functions

A transformer's feed-forward layer bends its signal through a nonlinearity. ReLU zeros out negatives; GELU and SiLU are its smooth cousins that let a little negative signal through. GELU is the default in GPT-style models. Feed the same five inputs through all three and watch how they differ on the negative side.

In [ ]:
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print(f'input: {x.tolist()}')
print(f'ReLU:  {F.relu(x).tolist()}')
print(f'GELU:  {[round(v, 3) for v in F.gelu(x).tolist()]}')
print(f'SiLU:  {[round(v, 3) for v in F.silu(x).tolist()]}')

**Explanation**

A side-by-side comparison harness - one input vector run through three activations so you can eyeball their shapes. The tell is the negative inputs: ReLU hard-clips them to 0, while GELU and SiLU let a small negative value survive.

**How the code works - step by step**
- **`x`** - five test values spanning negative to positive: `[-2, -0.5, 0, 0.5, 2]`.
- **`F.relu(x)`** - clamps everything below 0 to exactly 0.
- **`F.gelu(x)`** - smooth gating; small negatives become small negatives, not hard zeros.
- **`F.silu(x)`** - `x * sigmoid(x)`, another smooth curve with a slight negative dip.
- rounding in the prints keeps the smooth outputs readable.

**In one line:** ReLU cuts negatives to zero; GELU and SiLU soften the cut.

**What you'll see:** Four printed lines - the raw input, then ReLU (`[0, 0, 0, 0.5, 2]`), GELU, and SiLU. ReLU shows hard zeros for the two negative inputs; GELU and SiLU show small negative values there instead.

## 4 - Linear collapse: why activations matter

Here's the proof that nonlinearities aren't optional. Stack two linear layers with nothing between them and they collapse into a *single* matrix - so a hundred stacked linear layers have no more power than one. Drop a ReLU in the middle and that collapse breaks. This one demo is the reason every transformer block has an activation.

In [ ]:
import numpy as np
W1 = np.array([[1, 0], [1, 1]])
W2 = np.array([[2, 1], [0, 1]])
x  = np.array([1, 2])

# Two linear layers collapse to one matrix
print('two layers:', W2 @ (W1 @ x))     # [5 3]
print('combined:  ', (W2 @ W1) @ x)     # [5 3]  - identical

# A ReLU between them breaks the collapse
x2 = np.array([1, -3])
h  = np.maximum(0, W1 @ x2)
print('with ReLU: ', W2 @ h)            # [2 0]
print('collapsed: ', (W2 @ W1) @ x2)    # [0 -2]  - no single matrix matches

**Explanation**

A tiny proof-by-contradiction in NumPy, not a model. First it shows two linear layers equal their combined product (collapse). Then it inserts a ReLU and shows the combined single matrix no longer reproduces the result - the nonlinearity gave the network expressive power a single matrix cannot match.

**How the code works - step by step**
- **`W2 @ (W1 @ x)` vs `(W2 @ W1) @ x`** - both give `[5, 3]`: two linear layers are provably the same as one matrix `W2 @ W1`.
- **`h = maximum(0, W1 @ x2)`** - inserts a ReLU between the layers, clipping negatives to 0.
- **`W2 @ h` vs `(W2 @ W1) @ x2`** - now `[2, 0]` vs `[0, -2]`: no single matrix reproduces the ReLU version.

**In one line:** without a nonlinearity, depth is an illusion - the ReLU is what makes stacking worth anything.

**What you'll see:** Four lines: `two layers: [5 3]` and `combined: [5 3]` (identical - the collapse), then `with ReLU: [2 0]` and `collapsed: [0 -2]` (different - the collapse is broken).

## 5 - The full transformer attention block

Now assemble the real thing: multi-head self-attention plus a feed-forward network, exactly the unit that repeats dozens of times inside GPT and Llama. One `Linear` projects the input into Q, K, and V at once; the tensor is split across heads; PyTorch's fused causal attention does the scoring; heads are recombined; and a GELU feed-forward finishes the block. Input shape in, same shape out - that's what lets you stack it.

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=8):
        super().__init__()
        self.d_model = d_model; self.n_heads = n_heads; self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )
    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        return self.ffn(self.out(attn_out))

block = AttentionBlock(d_model=128, n_heads=8)
x = torch.randn(2, 16, 128)
out = block(x)
print(f'output shape: {out.shape}')

**Explanation**

A production-shaped PyTorch module, not a toy loop - it wires every earlier piece (matmul, attention, GELU) into one reusable `nn.Module`. The forward pass reshapes into heads, calls the fused `scaled_dot_product_attention` with `is_causal=True` (each token sees only itself and the past), merges heads back, then runs the feed-forward net. Shape is preserved so blocks chain.

**How the code works - step by step**
- **`__init__`** - defines `qkv` (one Linear producing 3x d_model), the `out` projection, and a `ffn` (Linear -> GELU -> Linear that expands to 4x width and back).
- **`self.qkv(x)` + `.chunk(3)`** - projects input once, then splits into Q, K, V.
- **`.view(...).transpose(1,2)`** - reshapes each into `(batch, n_heads, seq, d_head)` so heads attend independently.
- **`F.scaled_dot_product_attention(q,k,v, is_causal=True)`** - the fused attention from Skill 2, with a causal mask baked in.
- **`.transpose(1,2).contiguous().view(B,T,D)`** - stitches the heads back into one d_model vector per token.
- **`self.ffn(self.out(attn_out))`** - output projection then the position-wise feed-forward net.
- **driver** - builds a block, feeds a `(2, 16, 128)` batch, prints the output shape.

**In one line:** project to Q/K/V -> split into heads -> causal attention -> merge -> feed-forward, shape preserved.

**What you'll see:** One line: `output shape: torch.Size([2, 16, 128])` - identical to the input shape, confirming the block can be stacked without any reshaping between layers.

You now have the whole attention stack in your hands: matmul moves and mixes vectors, scaled dot-product attention turns that into a weighted lookup, activations bend the signal so stacked layers can't collapse into one matrix, and the AttentionBlock glues all of it into the exact unit that repeats N times inside GPT and Llama. The linear-collapse demo is the load-bearing intuition - it is *why* the GELU sits between the two feed-forward layers. Next stop (Module 4) is scaling this one block into a full model and training it.